In [11]:
import networkx as nx
import pylab
import plant_gravitropism as pg

# Observed arbor CSV
fname = "pimpi_Big4_D5_set1_day5_20191012_297_103_4_S.csv"
arbor = pg.rar.read_arbor_full(fname)

In [12]:
import numpy as np

# Fine alpha sampling around the spike
alphas = np.linspace(0.015, 0.055, 9)  # 0.015, 0.018, ..., 0.055
G = 1.0  # gravity

In [15]:
def show_error_work(arbor, alpha, G):
    print(f"\n=== Computing error for alpha={alpha:.5f}, G={G} ===")
    
    # Step 1: Generate optimized arbor
    results = pg.modified_arbor_best_cost(fname, G, alpha, 0)
    print(f"Number of optimized PQ lines: {len(results)}")
    
    # Step 2: Compute per-segment contribution to absolute and squared error
    abs_error_total = 0
    sq_error_total = 0
    
    for i, r in enumerate(results):
        # r[4], r[5] = observed point
        # r[6], r[7] = optimized PQ point
        obs_x, obs_y = r[4], r[5]
        opt_x, opt_y = r[6], r[7]
        dx = obs_x - opt_x
        dy = obs_y - opt_y
        abs_err = abs(dx) + abs(dy)  # depending on plant_gravitropism implementation
        sq_err = dx**2 + dy**2
        abs_error_total += abs_err
        sq_error_total += sq_err
        print(f"PQ line {i}: Obs=({obs_x:.3f},{obs_y:.3f}), Opt=({opt_x:.3f},{opt_y:.3f}), "
              f"|dx|+|dy|={abs_err:.5f}, dx^2+dy^2={sq_err:.5f}")
    
    print(f"Total absolute error: {abs_error_total:.5f}")
    print(f"Total squared error: {sq_error_total:.5f}")
    return abs_error_total, sq_error_total

In [16]:
for alpha in alphas:
    abs_err, sq_err = show_error_work(arbor, alpha, G)


=== Computing error for alpha=0.01500, G=1.0 ===
Number of optimized PQ lines: 6
PQ line 0: Obs=(6.693,5.013), Opt=(6.660,5.109), |dx|+|dy|=0.13012, dx^2+dy^2=0.01046
PQ line 1: Obs=(6.770,4.610), Opt=(6.645,5.001), |dx|+|dy|=0.51706, dx^2+dy^2=0.16883
PQ line 2: Obs=(6.770,4.610), Opt=(6.026,5.581), |dx|+|dy|=1.71617, dx^2+dy^2=1.49829
PQ line 3: Obs=(6.770,4.610), Opt=(6.581,4.880), |dx|+|dy|=0.45978, dx^2+dy^2=0.10899
PQ line 4: Obs=(6.693,5.013), Opt=(6.295,7.307), |dx|+|dy|=2.69313, dx^2+dy^2=5.42419
PQ line 5: Obs=(6.693,5.013), Opt=(6.473,6.788), |dx|+|dy|=1.99540, dx^2+dy^2=3.20059
Total absolute error: 7.51167
Total squared error: 10.41135

=== Computing error for alpha=0.02000, G=1.0 ===
Number of optimized PQ lines: 6
PQ line 0: Obs=(6.693,5.013), Opt=(6.660,5.109), |dx|+|dy|=0.13012, dx^2+dy^2=0.01046
PQ line 1: Obs=(6.770,4.610), Opt=(6.645,5.001), |dx|+|dy|=0.51706, dx^2+dy^2=0.16883
PQ line 2: Obs=(6.770,4.610), Opt=(6.026,5.581), |dx|+|dy|=1.71617, dx^2+dy^2=1.49829
PQ

In [17]:
def show_error_work_v2(fname, alpha, G):
    print(f"\n=== Computing error for alpha={alpha:.5f}, G={G} ===")
    
    G_graph = pg.rar.read_arbor_full(fname)
    G_opt = nx.Graph(Gravity=G)
    line_segs = pg.get_line_segments(G_graph)
    pg.graph_main_root(G_opt, line_segs)
    results = pg.modified_arbor_best_cost(fname, G, alpha, 0)
    pg.graph_opt_lines(G_opt, results)

    total_abs = 0
    total_sq = 0

    for i, result in enumerate(results):
        best_x, best_y = result[4], result[5]
        tip_x, tip_y   = result[6], result[7]
        lateral_tip = (tip_x, tip_y)
        main_root_pt = (best_x, best_y)

        print(f"\n--- Lateral root {i}: tip=({tip_x:.3f},{tip_y:.3f}), branch=({best_x:.3f},{best_y:.3f}) ---")

        # Get optimized curve coefficients
        b, c = pg.calc_coeff(G, best_x, best_y, tip_x, tip_y)
        print(f"  Optimized curve: f(x) = {G}*x^2 + {b:.5f}*x + {c:.5f}")

        # BFS to collect this lateral root's points
        lateral_points = []
        visited = set()
        queue = [lateral_tip]
        while queue:
            node = queue.pop(0)
            if node in visited:
                continue
            visited.add(node)
            label = G_graph.nodes[node]['label']
            if label in ('lateral root', 'lateral root tip'):
                lateral_points.append(node)
                for neighbor in G_graph.neighbors(node):
                    if neighbor not in visited and G_graph.nodes[neighbor]['label'] in ('lateral root', 'lateral root tip'):
                        queue.append(neighbor)

        print(f"  Found {len(lateral_points)} lateral root points:")
        abs_err = 0
        sq_err = 0
        for (obs_x, obs_y) in lateral_points:
            opt_y = G * obs_x**2 + b * obs_x + c
            err = obs_y - opt_y
            abs_err += abs(err)
            sq_err += err**2
            print(f"    obs=({obs_x:.3f},{obs_y:.3f}), opt_y={opt_y:.3f}, err={err:.5f}")

        print(f"  Subtotal abs={abs_err:.5f}, sq={sq_err:.5f}")
        total_abs += abs_err
        total_sq += sq_err

    print(f"\nTotal absolute error: {total_abs:.5f}")
    print(f"Total squared error:  {total_sq:.5f}")
    return total_abs, total_sq

# Run it around the spike
for alpha in [0.03, 0.045, 0.05, 0.055, 0.06]:
    show_error_work_v2(fname, alpha, G=1.0)


=== Computing error for alpha=0.03000, G=1.0 ===

--- Lateral root 0: tip=(6.660,5.109), branch=(6.693,5.013) ---
  Optimized curve: f(x) = 1.0*x^2 + -16.24199*x + 68.92503
  Found 2 lateral root points:
    obs=(6.660,5.109), opt_y=5.109, err=-0.00000
    obs=(6.717,5.147), opt_y=4.944, err=0.20289
  Subtotal abs=0.20289, sq=0.04116

--- Lateral root 1: tip=(6.645,5.001), branch=(6.718,4.882) ---
  Optimized curve: f(x) = 1.0*x^2 + -14.97612*x + 60.36040
  Found 2 lateral root points:
    obs=(6.645,5.001), opt_y=5.001, err=0.00000
    obs=(6.673,5.034), opt_y=4.954, err=0.07964
  Subtotal abs=0.07964, sq=0.00634

--- Lateral root 2: tip=(6.026,5.581), branch=(6.770,4.610) ---
  Optimized curve: f(x) = 1.0*x^2 + -14.10039*x + 54.23680
  Found 17 lateral root points:
    obs=(6.026,5.581), opt_y=5.581, err=-0.00000
    obs=(6.045,5.559), opt_y=5.542, err=0.01747
    obs=(6.080,5.524), opt_y=5.474, err=0.05077
    obs=(6.115,5.486), opt_y=5.406, err=0.08023
    obs=(6.168,5.433), opt_y